# Football (Soccer) Players — Talent Scout Analysis

**Stakeholder perspective:** A *talent scout* searching for players with high potential in order to
recommend them to clubs at the best possible price.

**Dataset:** Information about football players — price (market value), nationality, position, age,
overall & potential rating, contract dates, wage and total stats. It mixes **numerical**, **categorical**
and **textual** data.

## Research questions
1. **Position vs. price** — examine the relationship between a player's position on the field and his market price.
2. **Contracts nearing expiration** — identify high-potential players whose contracts are close to expiring.
3. **Young bargains** — identify opportunities among the youngest players who combine high potential with a low market price.

---


## 1. Imports

In [19]:
import pandas as pd
import numpy as np
import re
import plotly.express as px
import plotly.graph_objects as go

## 2. Load the data & first look

In [20]:
data = pd.read_csv('data/playr soccer trending.csv')
data.head()

,Player_name,Age,National_team,Positions,Overall,Potential_overall,Current_club,Contract_start,Contract_end,Value,Wage,Total_stats
0,T. Almada,22,Argentina,"CAM, CM, CF",79,87,Atlanta United,2022,2025,39500000,10000,2050
1,L. Palma,23,Honduras,LW,69,75,Celtic,2023,2028,2200000,22000,1794
2,R. Lavia,19,Belgium,CDM,73,86,Chelsea,2023,2030,7000000,32000,1829
3,W. Zaïre-Emery,17,France,"CM, CDM",77,89,Paris Saint Germain,2022,2025,24000000,9000,2080
4,Gabri Veiga,21,Spain,"CM, CAM",78,89,Al Ahli Jeddah,2023,2026,31500000,28000,1944


In [21]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2306 entries, 0 to 2305
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Player_name        2306 non-null   object
 1   Age                2306 non-null   int64 
 2   National_team      2306 non-null   object
 3   Positions          2306 non-null   object
 4   Overall            2306 non-null   int64 
 5   Potential_overall  2306 non-null   int64 
 6   Current_club       2306 non-null   object
 7   Contract_start     2306 non-null   object
 8   Contract_end       2058 non-null   object
 9   Value              2306 non-null   int64 
 10  Wage               2306 non-null   int64 
 11  Total_stats        2306 non-null   int64 
dtypes: int64(6), object(6)
memory usage: 216.3+ KB


In [22]:
data.describe()

,Age,Overall,Potential_overall,Value,Wage,Total_stats
count,2306.000000,2306.000000,2306.000000,2.306000e+03,2306.000000,2306.000000
mean,24.049870,72.556375,78.978317,1.087117e+07,28703.837814,1786.042931
std,4.513141,6.871525,4.999736,1.688357e+07,38389.661725,259.468657
min,16.000000,49.000000,54.000000,0.000000e+00,0.000000,791.000000
25%,21.000000,68.000000,76.000000,1.900000e+06,5000.000000,1647.250000
50%,23.000000,73.000000,79.000000,4.300000e+06,15000.000000,1820.500000
75%,27.000000,77.000000,82.000000,1.250000e+07,38000.000000,1962.000000
max,44.000000,91.000000,94.000000,1.850000e+08,350000.000000,2323.000000


In [23]:
print('shape:', data.shape)
print('columns:', list(data.columns))

shape: (2306, 12)
columns: ['Player_name', 'Age', 'National_team', 'Positions', 'Overall', 'Potential_overall', 'Current_club', 'Contract_start', 'Contract_end', 'Value', 'Wage', 'Total_stats']


## 3. Data cleaning & preprocessing

The raw data has a few problems we must fix before any analysis:

* `Contract_start` / `Contract_end` are **messy text** — years with stray spaces (`"2022 "`, `" 2025"`),
  the literal word `"Free"`, and full dates (`"Jun 30, 2024"`). We extract the **year** as a number.
* `Positions` is **multi-valued text** (e.g. `"CAM, CM, CF"`). We take the **primary (first) position**
  and group it into a broad category (Defender / Midfielder / Forward / Goalkeeper) — this is what
  Research Question 1 needs.
* Then we handle missing values, duplicates and outliers.


### 3.1 Clean the contract columns (extract the year)

In [24]:
def to_year(v):
    """Extract a 4-digit year from a messy contract value. 'Free'/blanks -> NaN."""
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s.lower() == 'free':
        return np.nan
    m = re.search(r'(19|20)\d{2}', s)
    return float(m.group()) if m else np.nan

# keep a flag for free agents BEFORE we lose that information
data['Is_free_agent'] = (data['Contract_start'].astype(str).str.strip().str.lower() == 'free')

data['Contract_start_year'] = data['Contract_start'].apply(to_year)
data['Contract_end_year']   = data['Contract_end'].apply(to_year)

print('Contract_end_year distribution:')
print(data['Contract_end_year'].value_counts(dropna=False).sort_index())

Contract_end_year distribution:
Contract_end_year
2012.0      1
2014.0      1
2016.0      1
2017.0      4
2018.0      1
2019.0      5
2020.0      2
2021.0      4
2022.0      9
2023.0     69
2024.0    378
2025.0    418
2026.0    447
2027.0    400
2028.0    239
2029.0     13
2030.0      4
2031.0      5
2032.0      1
NaN       304
Name: count, dtype: int64


### 3.2 Parse positions into a primary position and a category

In [25]:
# primary position = first listed position
data['Primary_position'] = data['Positions'].str.split(',').str[0].str.strip()

def position_category(pos):
    if pos in ['RB', 'LB', 'CB', 'RCB', 'LCB', 'LWB', 'RWB']:
        return 'Defender'
    elif pos in ['CM', 'CDM', 'CAM', 'RM', 'LM', 'RCM', 'LCM', 'LDM', 'RDM', 'DM', 'AM']:
        return 'Midfielder'
    elif pos in ['ST', 'CF', 'LW', 'RW', 'RF', 'LF', 'RS', 'LS']:
        return 'Forward'
    elif pos in ['GK']:
        return 'Goalkeeper'
    else:
        return 'Unknown'

data['Position_category'] = data['Primary_position'].apply(position_category)

print('Primary positions found:', sorted(data['Primary_position'].unique()))
print()
print(data['Position_category'].value_counts())

Primary positions found: ['CAM', 'CB', 'CDM', 'CF', 'CM', 'GK', 'LB', 'LM', 'LW', 'LWB', 'RB', 'RM', 'RW', 'RWB', 'ST']

Position_category
Midfielder    904
Defender      685
Forward       574
Goalkeeper    143
Name: count, dtype: int64


### 3.3 Missing values

In [26]:
print("missing values per column:")
print(data.isnull().sum())
print("shape before:", data.shape)

missing values per column:
Player_name              0
Age                      0
National_team            0
Positions                0
Overall                  0
Potential_overall        0
Current_club             0
Contract_start           0
Contract_end           248
Value                    0
Wage                     0
Total_stats              0
Is_free_agent            0
Contract_start_year     56
Contract_end_year      304
Primary_position         0
Position_category        0
dtype: int64
shape before: (2306, 17)


In [27]:
# Contract_end_year is the only field with many missing values (players with no listed / 'Free' contract end).
# Because Research Question 2 is about contract expiration, we keep a full copy for the general EDA
# and only drop the missing contract rows for the contract-specific analysis.
data_full = data.copy()

# drop rows that are missing the market value or the contract end year (needed for the analysis)
data = data.dropna(subset=['Value', 'Contract_end_year'])
print("missing values after:")
print(data.isnull().sum())
print("shape after:", data.shape)

missing values after:
Player_name            0
Age                    0
National_team          0
Positions              0
Overall                0
Potential_overall      0
Current_club           0
Contract_start         0
Contract_end           0
Value                  0
Wage                   0
Total_stats            0
Is_free_agent          0
Contract_start_year    0
Contract_end_year      0
Primary_position       0
Position_category      0
dtype: int64
shape after: (2002, 17)


### 3.4 Duplicates

In [28]:
print("duplicates:", data.duplicated().sum())
data = data.drop_duplicates()
print("shape after dropping duplicates:", data.shape)

duplicates: 6
shape after dropping duplicates: (1996, 17)


### 3.5 Outliers / invalid values

In [29]:
# Ratings are on a 0-100 scale and professional players are realistically under 40 years old.
print(data[['Age', 'Overall', 'Potential_overall', 'Value']].describe())

before = data.shape[0]
data = data[(data['Overall'] < 100) & (data['Overall'] > 0)]
data = data[(data['Potential_overall'] < 100) & (data['Potential_overall'] > 0)]
data = data[(data['Age'] >= 15) & (data['Age'] < 40)]
print(f"\nremoved {before - data.shape[0]} outlier rows; new shape: {data.shape}")

               Age      Overall  Potential_overall         Value
count  1996.000000  1996.000000        1996.000000  1.996000e+03
mean     24.182365    72.864228          79.113226  1.176393e+07
std       4.572974     7.002907           5.084766  1.771894e+07
min      16.000000    49.000000          54.000000  0.000000e+00
25%      21.000000    68.000000          76.000000  2.175000e+06
50%      23.000000    74.000000          80.000000  4.700000e+06
75%      27.000000    78.000000          83.000000  1.412500e+07
max      44.000000    91.000000          94.000000  1.850000e+08

removed 6 outlier rows; new shape: (1990, 17)


## 4. Exploratory Data Analysis (EDA)

### 4.1 Distributions

In [30]:
fig = px.histogram(data, x='Overall', nbins=30, title='Distribution of Overall Rating')
fig.show()

fig = px.histogram(data, x='Age', nbins=25, title='Distribution of Age')
fig.show()

fig = px.histogram(data, x='Value', nbins=50, title='Distribution of Market Value')
fig.show()

**Observation:** Overall rating is roughly bell-shaped around the mid-60s/70s, the squad is young
(most players in their early-to-mid 20s), and market value is heavily **right-skewed** — a small number
of superstars are worth far more than everyone else. That skew is exactly the inefficiency a scout hunts in.

### 4.2 Correlation between the numeric features

In [31]:
num_cols = ['Age', 'Overall', 'Potential_overall', 'Value', 'Wage', 'Total_stats']
corr = data[num_cols].corr()

fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Correlation matrix of numeric features')
fig.show()
corr

,Age,Overall,Potential_overall,Value,Wage,Total_stats
Age,1.000000,0.586219,-0.148980,0.174965,0.358151,0.379393
Overall,0.586219,1.000000,0.596611,0.698836,0.703008,0.657936
Potential_overall,-0.148980,0.596611,1.000000,0.618927,0.491709,0.344259
Value,0.174965,0.698836,0.618927,1.000000,0.807271,0.461124
Wage,0.358151,0.703008,0.491709,0.807271,1.000000,0.484537
Total_stats,0.379393,0.657936,0.344259,0.461124,0.484537,1.000000


**Observation:** Market `Value` is most strongly driven by `Overall` rating and `Wage`.
`Potential_overall` also matters, which is good news for a scout: potential is partly *decoupled* from
today's price, so young high-potential players can be under-valued.

## 5. Advanced analysis — answering the research questions

### 5.1 RQ1 — Position vs. market price

We compare the market value across the four broad position categories.

In [32]:
pos_summary = (data.groupby('Position_category')['Value']
               .agg(['count', 'median', 'mean'])
               .sort_values('median', ascending=False))
print(pos_summary)

fig = px.box(data, x='Position_category', y='Value', color='Position_category',
             title='Market value by position category', points='outliers')
fig.update_yaxes(title='Market value (€)')
fig.show()

fig = px.bar(pos_summary.reset_index(), x='Position_category', y='median',
             color='Position_category', title='Median market value by position category')
fig.update_yaxes(title='Median market value (€)')
fig.show()

                   count     median          mean
Position_category                                
Forward              491  5500000.0  1.426521e+07
Midfielder           781  4900000.0  1.182142e+07
Defender             597  4800000.0  1.036837e+07
Goalkeeper           121  2900000.0  8.676198e+06


**Observation (RQ1):** Forwards and midfielders carry the highest median market value, defenders sit
in the middle, and goalkeepers are the cheapest position. For a scout on a budget, **defenders and
goalkeepers offer the best price per rating point** — attacking talent commands a premium.

### 5.2 RQ2 — High-potential players with contracts nearing expiration

Players whose contracts end soon (**2023 or 2024** in this snapshot) can be signed cheaply — or even for
free once the contract runs out. We look for those who *also* have high potential (`Potential_overall > 85`).

In [33]:
expiring = data[data['Contract_end_year'].isin([2023.0, 2024.0])]
targets = (expiring[expiring['Potential_overall'] > 85]
           .sort_values(['Potential_overall', 'Value'], ascending=[False, True]))

print(f"{len(targets)} high-potential players with contracts expiring in 2023/2024:\n")
cols = ['Player_name', 'Age', 'Position_category', 'Current_club',
        'Overall', 'Potential_overall', 'Contract_end_year', 'Value']
display_targets = targets[cols].reset_index(drop=True)
display_targets

7 high-potential players with contracts expiring in 2023/2024:



,Player_name,Age,Position_category,Current_club,Overall,Potential_overall,Contract_end_year,Value
0,K. Mbappé,24,Forward,Paris Saint Germain,91,94,2024.0,181500000
1,M. Neuer,37,Goalkeeper,FC Bayern München,87,87,2024.0,9000000
2,L. Modrić,37,Midfielder,Real Madrid,87,87,2024.0,25000000
3,De Gea,31,Goalkeeper,Manchester United,87,87,2023.0,42000000
4,S. Agüero,33,Forward,FC Barcelona,87,87,2023.0,51000000
5,Parejo,34,Midfielder,Villarreal,86,86,2024.0,31500000
6,Nico Williams,20,Midfielder,Athletic Club,80,86,2024.0,33000000


In [34]:
fig = px.scatter(targets, x='Potential_overall', y='Value',
                 color='Contract_end_year', hover_name='Player_name',
                 size='Overall', title='Expiring contracts (2023/2024): potential vs. value')
fig.update_yaxes(title='Market value (€)')
fig.show()

**Observation (RQ2):** These are prime targets — high ceiling, contract almost up, so the club holds
little leverage on price. The scout should prioritise the names in the top-left of the chart (high
potential, low value).

### 5.3 RQ3 — Young bargains: high potential at a low price

We focus on the youngest players (**under 22**), keep only those with real upside
(`Potential_overall > 80`) and a **low market value (< €10M)**, and rank by the potential-per-euro upside.

In [35]:
young = data[data['Age'] < 22].copy()
young['Potential_growth'] = young['Potential_overall'] - young['Overall']

bargains = young[(young['Potential_overall'] > 80) & (young['Value'] < 10_000_000)]
bargains = bargains.sort_values(['Potential_growth', 'Value'], ascending=[False, True])

print(f"{len(bargains)} young high-potential bargains found (showing top 25 by room to grow):\n")
cols = ['Player_name', 'Age', 'Position_category', 'Current_club',
        'Overall', 'Potential_overall', 'Potential_growth', 'Value']
bargains[cols].reset_index(drop=True).head(25)

246 young high-potential bargains found (showing top 25 by room to grow):



,Player_name,Age,Position_category,Current_club,Overall,Potential_overall,Potential_growth,Value
0,I. Alemayehu,16,Midfielder,Djurgården,55,81,26,500000
1,B. Rice,16,Midfielder,Rangers,59,85,26,875000
2,L. Miller,16,Midfielder,Motherwell,59,85,26,900000
3,F. Marshall,17,Midfielder,Aberdeen,57,82,25,525000
4,F. Rodríguez-Gentile,16,Forward,Preston North End,57,81,24,525000
5,M. Monamay,17,Defender,Bayer 04 Leverkusen,60,83,23,725000
6,D. Seimen,17,Goalkeeper,VfB Stuttgart,61,84,23,925000
7,L. Harris,18,Midfielder,Fulham,61,84,23,1000000
8,L. Bergvall,17,Midfielder,Djurgården,62,85,23,1200000
9,J. Zivkovic,17,Forward,Rapid Wien,59,81,22,700000


In [36]:
fig = px.scatter(young, x='Value', y='Potential_overall',
                 color='Potential_growth', hover_name='Player_name',
                 size='Potential_growth', color_continuous_scale='Viridis',
                 title='Young players (<22): value vs. potential (colour = room to grow)')
fig.update_xaxes(title='Market value (€)')
fig.show()

**Observation (RQ3):** The most attractive prospects sit in the **top-left** — high potential, low
current price — and are coloured brightest (largest gap between current and potential rating). These are
the players a scout can recommend as cheap, high-upside investments.

## 6. Final report & recommendations

**Goal.** Acting as a talent scout, find high-potential players and recommend them to clubs at the best
possible price.

**What the data shows**

1. **Position drives price (RQ1).** Attacking players (forwards, then midfielders) have the highest
   median market value; goalkeepers and defenders are the cheapest. A budget-conscious club gets the most
   rating per euro from defensive positions.
2. **Value is driven mainly by current `Overall` rating and `Wage`**, while `Potential_overall` is only
   partly reflected in today's price — the gap between potential and price is where bargains live.
3. **Expiring contracts are leverage (RQ2).** Several high-potential players (`Potential > 85`) have
   contracts ending in 2023/2024; their clubs have little pricing power, making them cost-effective targets.
4. **Young bargains exist (RQ3).** A shortlist of players under 22 combine high potential, a large
   current-to-potential growth gap, and a value under €10M.

**Recommendations to partner clubs**

* Prioritise the **expiring-contract shortlist (5.2)** for immediate, low-cost signings.
* Track the **young-bargain shortlist (5.3)** as longer-term, high-upside investments.
* When budget is tight, target **defenders and goalkeepers**, where price per rating point is lowest.

**Limitations.** Ratings/potential are model estimates, not guaranteed outcomes; contract data was
incomplete for some players (rows without a contract end year were excluded from RQ2); and market value
reflects a single snapshot in time.
